In [1]:
import kagglehub
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'Is working on CUDA ? : {torch.cuda.is_available()}')

path = kagglehub.dataset_download("mitgandhi10/dataset-for-cnn")

print(f"path to dataset : {path}")

lungs_folder = os.path.join(path, "xray_dataset_covid19")


Is working on CUDA ? : True
path to dataset : C:\Users\Lacha\.cache\kagglehub\datasets\mitgandhi10\dataset-for-cnn\versions\2


In [2]:
from torchvision import datasets, transforms

lungs_folder_train = os.path.join(lungs_folder, "train")
lungs_folder_test = os.path.join(lungs_folder, "test")


train_set = datasets.ImageFolder(lungs_folder_train, transform=transforms)
test_set= datasets.ImageFolder(lungs_folder_test, transform=transforms)

In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, 
                          batch_size = 32,
                          shuffle=True)

test_loader = DataLoader(test_set, 
                         batch_size=32, 
                         shuffle=32)

In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Définition explicite de la variable (on l'appelle `mes_transformations` pour éviter toute confusion avec le module)
mes_transformations = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Recréation des datasets en passant bien `mes_transformations`
train_dataset = datasets.ImageFolder(root=lungs_folder_train, transform=mes_transformations)
test_dataset = datasets.ImageFolder(root=lungs_folder_test, transform=mes_transformations)

# 3. Recréation des DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [5]:
import torch.nn as nn

class DeepPneumoniaCNN(nn.Module):
    def __init__(self):
        super(DeepPneumoniaCNN, self).__init__()
        
        # --- L'EXTRACTEUR DE CARACTÉRISTIQUES (5 Blocs) ---
        # On utilise nn.Sequential pour garder le code propre
        self.features = nn.Sequential(
            # Bloc 1 : 224x224 -> 112x112
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), # Accélère l'entraînement sur GPU
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloc 2 : 112x112 -> 56x56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloc 3 : 56x56 -> 28x28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloc 4 : 28x28 -> 14x14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloc 5 : 14x14 -> 7x7
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        # --- LE CLASSIFIEUR (Avec Dropout pour l'effet Bagging) ---
        self.classifier = nn.Sequential(
            nn.Flatten(),
            
            # La taille de l'image est passée de 224 à 7 après 5 divisions par 2
            # 512 canaux * 7 * 7 = 25088 paramètres en entrée
            nn.Linear(512 * 7 * 7, 512),
            nn.ReLU(),
            
            # "Bagging" interne : on éteint 50% des neurones aléatoirement à chaque batch
            nn.Dropout(p=0.5), 
            
            # 1 SEUL NEURONE en sortie car on veut une seule probabilité
            nn.Linear(512, 1),
            
            # Fonction Sigmoïde pour écraser la valeur entre 0.0 et 1.0
            nn.Sigmoid() 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instanciation sur la RTX 5060
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepPneumoniaCNN().to(device)

In [6]:
def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=10):
    """
    Fonction principale pour optimiser le modèle.
    """
    # Historique pour potentiellement tracer des graphiques plus tard
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    
    for epoch in range(num_epochs):
        print(f"--- Epoch {epoch+1}/{num_epochs} ---")
        
        # 1. PHASE D'ENTRAÎNEMENT
        model.train() # Active le Dropout et la Batch Normalization
        train_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images = images.to(device)
            # CRUCIAL POUR BCELoss :
            # Les labels arrivent comme [0, 1, 1, 0].
            # On les convertit en Float (0.0, 1.0) et on ajoute une dimension pour 
            # matcher la sortie du modèle qui sera de la forme [[0.1], [0.9], [0.8], [0.2]]
            labels = labels.to(device).float().unsqueeze(1)
            
            # Remise à zéro des gradients de la boucle précédente
            optimizer.zero_grad()
            
            # Prédiction (Forward pass)
            outputs = model(images)
            
            # Calcul de l'erreur
            loss = criterion(outputs, labels)
            
            # Rétropropagation (Backward pass)
            loss.backward()
            
            # Mise à jour des poids
            optimizer.step()
            
            # --- Calcul des statistiques d'entraînement ---
            train_loss += loss.item() * images.size(0)
            
            # Comme la sortie est une probabilité entre 0 et 1 (Sigmoid), 
            # tout ce qui est >= 0.5 est considéré comme prédiction '1' (Pneumonie)
            predicted = (outputs >= 0.5).float()
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        epoch_train_loss = train_loss / total_train
        epoch_train_acc = correct_train / total_train
        
        # 2. PHASE DE TEST (ÉVALUATION)
        # On évalue le modèle sur le jeu de test à la fin de chaque epoch
        epoch_test_loss, epoch_test_acc = evaluate_model(model, test_loader, criterion)
        
        # Sauvegarde dans l'historique
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['test_loss'].append(epoch_test_loss)
        history['test_acc'].append(epoch_test_acc)
        
        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"Test Loss:  {epoch_test_loss:.4f} | Test Acc:  {epoch_test_acc:.4f}\n")
        
    return history


def evaluate_model(model, dataloader, criterion):
    """
    Fonction pour tester les caractéristiques (performances) du modèle 
    sans modifier ses poids.
    """
    model.eval() # Désactive le Dropout et fige la Batch Normalization
    running_loss = 0.0
    correct = 0
    total = 0
    
    # torch.no_grad() coupe le moteur de calcul des gradients : 
    # cela divise l'utilisation de la VRAM de votre GPU par 2 et accélère le test
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            # Même redimensionnement critique que pour l'entraînement
            labels = labels.to(device).float().unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            predicted = (outputs >= 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    test_loss = running_loss / total
    test_acc = correct / total
    
    return test_loss, test_acc

In [7]:

criterion = nn.BCELoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Lancement de l'entraînement sur 15 epochs
historique = train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=15)

--- Epoch 1/15 ---
Train Loss: 8.6646 | Train Acc: 0.6081
Test Loss:  0.8454 | Test Acc:  0.7750

--- Epoch 2/15 ---
Train Loss: 5.4586 | Train Acc: 0.8851
Test Loss:  50.0000 | Test Acc:  0.5000

--- Epoch 3/15 ---
Train Loss: 2.4988 | Train Acc: 0.9392
Test Loss:  50.0000 | Test Acc:  0.5000

--- Epoch 4/15 ---
Train Loss: 1.1153 | Train Acc: 0.9595
Test Loss:  50.0000 | Test Acc:  0.5000

--- Epoch 5/15 ---
Train Loss: 1.7767 | Train Acc: 0.9662
Test Loss:  50.0000 | Test Acc:  0.5000

--- Epoch 6/15 ---
Train Loss: 1.8236 | Train Acc: 0.9595
Test Loss:  15.7343 | Test Acc:  0.7750

--- Epoch 7/15 ---
Train Loss: 1.7734 | Train Acc: 0.9730
Test Loss:  10.9249 | Test Acc:  0.7750

--- Epoch 8/15 ---
Train Loss: 1.4324 | Train Acc: 0.9797
Test Loss:  5.2358 | Test Acc:  0.9250

--- Epoch 9/15 ---
Train Loss: 1.2855 | Train Acc: 0.9595
Test Loss:  5.4849 | Test Acc:  0.9000

--- Epoch 10/15 ---
Train Loss: 3.5951 | Train Acc: 0.9595
Test Loss:  16.2141 | Test Acc:  0.7500

--- Epoch 11